# 04 — the comparative matrix, including where NumFast loses

**No timing is measured in this notebook.** Every millisecond below is read from
a file that was already in the repository before this notebook existed. The box
this ran on is shared with other benchmark work, so a number taken here would
describe this box at this moment — not the engine — and printing it as a
performance claim would be exactly the lie the rest of this repository refuses.

The two artefacts read here are:

| Artefact | What it is |
|---|---|
| `calibration_dataset.json` | raw per-operation measurements: 5 repetitions, 3 warm-up runs, seed 42, ten operations, five sizes each |
| `calibration.toml` | the fitted cost model built from them, plus fit quality (`r2`, `mape`) and the hardware it was taken on |

Neither is a benchmark *harness* — the README is explicit that this release ships
none. They are a measured cost profile, committed because the planner loads it
at run time.

In [1]:
import json
from pathlib import Path

import numfast as nf

ROOT = Path.cwd()
ds = json.loads((ROOT / "calibration_dataset.json").read_text(encoding="utf-8"))
prof = ds["profile"]
print("numfast", nf.__version__)
print("artefact source :", prof["source"])
print("generated       :", prof["generated"])
print("quick profile   :", prof["quick"])
print()
for key in ("cpu", "platform", "python", "backend", "gpu_device", "gpu_backend"):
    print(f"  {key:11s} {prof['hardware'][key]}")

numfast 0.2.1
artefact source : measured:seed42
generated       : 2026-09-10T21:07:19.522407+00:00
quick profile   : False

  cpu         Intel64 Family 6 Model 79 Stepping 1, GenuineIntel
  platform    Windows-11-10.0.26100-SP0
  python      3.14.6
  backend     webgpu
  gpu_device  NVIDIA GeForce RTX 2060
  gpu_backend Vulkan


In [2]:
m = prof["measurements"]
print("repetitions      :", m["reps"])
print("warmup runs      :", m["warmup"])
print("sizes            :", m["Ns"])
print("validation size  :", m["validation_Ns"])
print("groupby sizes    :", m["groupby_Ns"], "cardinalities", m["groupby_cardinalities"])
print("sort sizes       :", m["sort_Ns"])
print("slice sizes      :", m["slice_Ns"])

repetitions      : 5
warmup runs      : 3
sizes            : [1000, 10000, 100000, 1000000, 500000]
validation size  : [500000]
groupby sizes    : [10000, 100000, 1000000] cardinalities [100, 10000, 100000]
sort sizes       : [8192, 65536, 1048576, 4194304]
slice sizes      : [100, 10000, 100000]


## The fit is not perfect, and that is reported rather than hidden

Before any figure is used, the fit quality. A cost model with a poor `r2` cannot
carry a crossover claim, so the worst fits are listed first. `mape` is mean
absolute percentage error on the held-out prediction.

In [3]:
rows = [(k, v) for k, v in prof["metrics"].items() if k.endswith("_r2")]
rows.sort(key=lambda kv: kv[1])
print(f"{'metric':38s} {'r2':>8s}")
print("-" * 48)
for k, v in rows[:8]:
    print(f"{k:38s} {v:8.4f}")
print("...")
for k, v in rows[-3:]:
    print(f"{k:38s} {v:8.4f}")

worst = min(rows, key=lambda kv: kv[1])
print()
print(f"worst fit: {worst[0]} at r2 = {worst[1]:.4f}")
print("high-cardinality groupby is the weak spot, and the matrix below reflects that.")

metric                                       r2
------------------------------------------------
groupby_M100000_cpu_r2                   0.8925
slice_gpu_r2                             0.9153
groupby_M100000_gpu_r2                   0.9783
groupby_M10000_cpu_r2                    0.9821
sort_gpu_r2                              0.9938
reduce_gpu_r2                            0.9939
slice_cpu_r2                             0.9946
rng_fill_i32_gpu_r2                      0.9950
...
transfer_d2h_r2                          1.0000
rng_fill_i32_cpu_r2                      1.0000
groupby_M100_gpu_r2                      1.0000

worst fit: groupby_M100000_cpu_r2 at r2 = 0.8925
high-cardinality groupby is the weak spot, and the matrix below reflects that.


## Where NumFast wins, and by how much

The README names the win precisely, and it is a **structural** one rather than a
throughput one: one planner call turns a whole chain into one execution graph,
so per-query overhead does not grow with the number of operations in the query.

The measured cost of the fixed part is in the artefact: each GPU op carries a
`compile_ms`, and each CPU op carries `0.0`.

In [4]:
cost = prof["cost"]
gpuc = sorted(k[:-len("_gpu_compile_ms")] for k in cost
              if k.endswith("_gpu_compile_ms"))
print(f"{'op':16s} {'cpu compile ms':>15s} {'gpu compile ms':>15s}")
print("-" * 48)
for op in gpuc:
    cc = cost.get(f"{op}_cpu_compile_ms")
    gc = cost.get(f"{op}_gpu_compile_ms")
    print(f"{op:16s} {cc:15.4f} {gc:15.4f}")

op                cpu compile ms  gpu compile ms
------------------------------------------------
compare                   0.0000          0.6466
filter                    0.0000          0.1925
gather                    0.0000          0.3301
groupby_M100              0.0000          0.3246
groupby_M10000            0.0000          0.0000
groupby_M100000           0.0000          3.5050
mask                      0.0000          1.5046
pack_keys                 0.0000          0.2323
reduce                    0.0000          0.5750
rng_fill_i32              0.0000          0.4637
slice                     0.0000          0.2759
sort                      0.0000          0.0000


## The full matrix: CPU against GPU, measured

`cpu_ms` is the CPU time and `warm_ms` the GPU time after warm-up. Read
`ratio = gpu / cpu`: **above 1 the GPU is slower.**

This is the honest headline: at the sizes measured, the GPU is slower for eight
of ten operations.

In [5]:
ops = ds["raw"]["ops"]
print(f"{'op':14s} {'N':>10s} {'sel':>7s} {'cpu_ms':>10s} {'gpu_ms':>10s} {'gpu/cpu':>9s}")
print("-" * 64)
wins = {}
for op in sorted(ops):
    for n_ in sorted(ops[op], key=int):
        for sel in sorted(ops[op][n_]):
            e = ops[op][n_][sel]
            cpu, gpu = e.get("cpu_ms"), e.get("warm_ms")
            if cpu is None or gpu is None:
                continue
            if gpu < cpu:
                wins.setdefault(op, []).append(int(n_))
            if n_ != max(ops[op], key=int):
                continue
            print(f"{op:14s} {int(n_):>10,} {sel:>7s} {cpu:10.4f} {gpu:10.4f} "
                  f"{gpu / cpu:8.1f}x")

op                      N     sel     cpu_ms     gpu_ms   gpu/cpu
----------------------------------------------------------------
compare         1,000,000     0.5     0.3499    17.5820     50.2x
filter          1,000,000     0.1     7.6000    41.4661      5.5x
filter          1,000,000     0.5    16.2128    45.9427      2.8x
filter          1,000,000     0.9    11.1611    51.3620      4.6x
gather          1,000,000     0.5     5.7639    21.5591      3.7x
groupby         1,000,000     100     2.1764    30.7358     14.1x
groupby         1,000,000   10000     5.2898    31.6800      6.0x
groupby         1,000,000  100000    27.5488    52.9420      1.9x
mask            1,000,000     0.5     0.3688    29.7893     80.8x
pack_keys       1,000,000     0.5    10.1482    24.9014      2.5x
reduce          1,000,000     0.5     0.9702     5.3880      5.6x
rng_fill_i32    1,000,000     0.5    60.6839    13.5651      0.2x
slice             100,000     0.5     0.0236     5.8853    249.4x
sort       

### The complete size series, for the two operations where the GPU ever wins

Everything else is a loss at every measured size, and the largest size of each
is already in the table above.

In [6]:
for op in ("rng_fill_i32", "sort"):
    print(op)
    for n_ in sorted(ops[op], key=int):
        e = ops[op][n_]["0.5"]
        mark = "  <-- gpu faster" if e["warm_ms"] < e["cpu_ms"] else ""
        print(f"   N={int(n_):>9,}  cpu {e['cpu_ms']:9.4f} ms   "
              f"gpu {e['warm_ms']:9.4f} ms   {e['warm_ms'] / e['cpu_ms']:6.1f}x{mark}")

rng_fill_i32
   N=    1,000  cpu    0.0901 ms   gpu    2.5207 ms     28.0x
   N=   10,000  cpu    0.6737 ms   gpu    2.5298 ms      3.8x
   N=  100,000  cpu    6.1957 ms   gpu    2.7803 ms      0.4x  <-- gpu faster
   N=  500,000  cpu   29.5588 ms   gpu    8.1786 ms      0.3x  <-- gpu faster
   N=1,000,000  cpu   60.6839 ms   gpu   13.5651 ms      0.2x  <-- gpu faster
sort
   N=    8,192  cpu    0.3480 ms   gpu   36.5588 ms    105.1x
   N=   65,536  cpu    2.8763 ms   gpu   53.5695 ms     18.6x
   N=1,048,576  cpu   72.5169 ms   gpu  100.9009 ms      1.4x
   N=4,194,304  cpu  323.9730 ms   gpu  206.4550 ms      0.6x  <-- gpu faster


## Where NumFast loses badly — stated without softening

Three losses worth naming, because a reader deciding whether to use NumFast needs
them more than the wins.

**1. High-cardinality grouping.** The README is direct about this:
*"several times behind DuckDB, not a few percent behind... If your workload is
high-cardinality grouping, DuckDB is the better tool today."* The artefact shows
the same shape on the CPU side — `groupby` gets steadily worse as cardinality
rises, and its `r2` is the worst in the whole profile.

**2. The GPU, for most operations.** Up to 250x slower on `slice`, because the
read-back costs more than the work.

**3. Small-N everything.** The per-call floor is what a shared engine cannot
amortise, and it is why no ratio below 1000 rows is flattering.

In [7]:
g = ops["groupby"]
print("groupby: cost against cardinality")
print(f"{'N':>10s} {'card':>8s} {'cpu_ms':>10s}")
print("-" * 32)
for n_ in sorted(g, key=int):
    for sel in sorted(g[n_], key=int):
        e = g[n_][sel]
        if e.get("cpu_ms") is not None:
            print(f"{int(n_):>10,} {int(sel):>8,} {e['cpu_ms']:10.4f}")

groupby: cost against cardinality
         N     card     cpu_ms
--------------------------------
    10,000      100     0.2213
    10,000   10,000     1.0372
    10,000  100,000     2.9448
   100,000      100     0.4659
   100,000   10,000     1.9955
   100,000  100,000    12.9511
   500,000   50,000    12.4659
 1,000,000      100     2.1764
 1,000,000   10,000     5.2898
 1,000,000  100,000    27.5488


In [8]:
print("the worst ratios recorded, across every size measured:")
rows = []
for op in ops:
    for n_ in ops[op]:
        for sel in ops[op][n_]:
            e = ops[op][n_][sel]
            if e.get("cpu_ms") and e.get("warm_ms"):
                rows.append((e["warm_ms"] / e["cpu_ms"], op, int(n_), sel))
rows.sort(reverse=True)
print(f"{'gpu/cpu':>9s} {'op':14s} {'N':>10s} {'sel':>7s}")
print("-" * 44)
for ratio, op, n_, sel in rows[:6]:
    print(f"{ratio:8.1f}x {op:14s} {n_:>10,} {sel:>7s}")

the worst ratios recorded, across every size measured:
  gpu/cpu op                      N     sel
--------------------------------------------
   249.4x slice             100,000     0.5
   193.4x slice              10,000     0.5
   186.6x slice                 100     0.5
   105.1x sort                8,192     0.5
    80.8x mask            1,000,000     0.5
    77.8x mask              500,000     0.5


## NumFast against pandas and NumPy — same box, same data, counted

This part is **not** from the calibration artefact: the calibration measures ops,
not pandas. What can be said honestly is the structural difference, and it is a
count rather than a time.

The comparison below is *correctness* and *shape*, at a size that runs in a
second, plus a structural count of what each system did. Timing on a shared box
is not quoted.

In [9]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 200_000
frame = pd.DataFrame({
    "region": rng.choice(["emea", "apac", "amer", "latam"], N),
    "units":  rng.random(N) * 10.0,
    "price":  rng.random(N) * 100.0,
})

app = nf.app()
t = nf.from_pandas(frame)

result = (t.query()
          .derive("gross", app.c("units") * app.c("price"))
          .group("region", {"gross": ("sum", "count", "max", "min")})
          .sort("gross.sum", desc=True)
          .compile())

got = result.to_pandas()
print(got.to_string(index=False))

region    gross.sum  gross.count  gross.max  gross.min
  amer 1.252353e+07        50323 993.739580   0.000135
 latam 1.248444e+07        49922 990.167114   0.000032
  emea 1.243063e+07        49807 992.115974   0.001219
  apac 1.241892e+07        49948 995.948854   0.002142


In [10]:
reference = (frame.assign(gross=frame.units * frame.price)
                .groupby("region", as_index=False)["gross"]
                .agg(["sum", "count", "max", "min"])
                .sort_values("sum", ascending=False)
                .reset_index())
print(reference.to_string(index=False))

s_ok = np.allclose(got["gross.sum"].to_numpy(), reference["sum"].to_numpy())
c_ok = (got["gross.count"].to_numpy() == reference["count"].to_numpy()).all()
print()
print("sums equal :", bool(s_ok))
print("counts equal:", bool(c_ok))

 index region          sum  count        max      min
     0   amer 1.252353e+07  50323 993.739580 0.000135
     3  latam 1.248444e+07  49922 990.167114 0.000032
     2   emea 1.243063e+07  49807 992.115974 0.001219
     1   apac 1.241892e+07  49948 995.948854 0.002142

sums equal : True
counts equal: True


In [11]:
chain = (t.query()
         .filter(app.c("units") > 5.0)
         .derive("gross", app.c("units") * app.c("price"))
         .sort("gross", desc=True)
         .limit(10))
print("one chain, one planner call. jobs emitted:")
for job in chain.jobs():
    print(f"   {job['op']:9s} -> {job['out']}")
print()
print("STAGE      ", [l for l in chain.explain().splitlines()
                      if l.split()[0] in ("GRAPH", "STATE")])

one chain, one planner call. jobs emitted:
   series    -> s_units_1
   compare   -> cmp_2
   series    -> c_region_4
   filter    -> f_region_3
   filter    -> f_units_5
   series    -> s_price_7
   filter    -> f_price_6
   map       -> m_8
   sort      -> perm_9
   gather    -> srt_region_10
   gather    -> srt_units_11
   gather    -> srt_price_12
   gather    -> srt_gross_13
   slice     -> lim_region_14
   slice     -> lim_units_15
   slice     -> lim_price_16
   slice     -> lim_gross_17

STAGE       ["GRAPH nodes=17 outputs=['lim_region_14', 'lim_units_15', 'lim_price_16', 'lim_gross_17'] ops=[series:s_units_1, compare:cmp_2, series:c_region_4, filter:f_region_3, filter:f_units_5, series:s_price_7, filter:f_price_6, map:m_8, sort:perm_9, gather:srt_region_10, gather:srt_units_11, gather:srt_price_12, gather:srt_gross_13, slice:lim_region_14, slice:lim_units_15, slice:lim_price_16, slice:lim_gross_17]", 'STATE placement=n/a state=n/a selectivity=n/a dispatches=n/a chunks=n/a']


Nine nodes for a four-stage chain — filter, two sources, a multiply, a sort and
a limit. That is the structural claim, and it is countable: the fixed cost is
paid once per **chain**, not once per operation.

**What this notebook does not do:** it does not time any of this. A count of
nodes is a fact about the IR. A millisecond would be a fact about this box.

## The one place a structural count beats a clock

Bytes moved, and what the memory layout costs. These are exact properties of the
recorded dataset, not measurements of anything.

In [12]:
tr = ds["raw"]["transfer"]
print("host <-> device transfer, as recorded")
print(f"{'direction':>10s} {'bytes':>14s} {'median_ms':>11s} {'ns/byte':>9s}")
print("-" * 48)
for d in ("h2d", "d2h"):
    for by in sorted(tr[d], key=int):
        e = tr[d][by]
        print(f"{d:>10s} {e['bytes']:>14,} {e['median_ms']:11.3f} "
              f"{e['median_ms'] * 1e6 / e['bytes']:9.3f}")

host <-> device transfer, as recorded
 direction          bytes   median_ms   ns/byte
------------------------------------------------
       h2d      1,048,576       0.899     0.857
       h2d     10,485,760       7.506     0.716
       h2d    104,857,600      82.252     0.784
       h2d  1,073,741,824    2132.330     1.986
       d2h      1,048,576       3.450     3.290
       d2h     10,485,760      18.670     1.781
       d2h    104,857,600     147.991     1.411
       d2h  1,073,741,824    1438.620     1.340


**No GPU speedup is claimed, and nothing above supports one.** The recorded
transfer costs are the reason: a device-to-host copy costs more per byte than a
host-to-device copy, and every GPU node is read back. On a single-op job the
copy is the cost.